In [1]:
import os
import pandas as pd
from pinecone import Pinecone
import google.generativeai as genai
from tqdm import tqdm
import uuid
from dotenv import load_dotenv
import time

load_dotenv()

# env
GEMINI_KEY = os.getenv("GOOGLE_API_KEY")
PINECONE_KEY = os.getenv("PINECONE_API_KEY")
INDEX_NAME = os.getenv("INDEX_NAME")

# setup
genai.configure(api_key=GEMINI_KEY)
pc = Pinecone(api_key=PINECONE_KEY)
index = pc.Index(INDEX_NAME)

# ------------------------------
# STEP 1: LOAD CSV
# ------------------------------
CSV_PATH = "iot-ase/backend/iot_intrusion_reduced.csv"
df = pd.read_csv(CSV_PATH)
print(f"📌 Loaded {len(df)} rows from CSV\n")


# ------------------------------
# GET LAST INDEXED ROW
# ------------------------------
def get_last_indexed_row():
    """
    We store row index inside metadata on first run.
    But since your old script didn't store it,
    we use Pinecone vector count as fallback.
    """
    stats = index.describe_index_stats()
    total = stats.get("total_vector_count", 0)

    if total == 0:
        return 0
    
    # Prevent going out of range
    if total >= len(df):
        return len(df)

    return total


start_from = get_last_indexed_row()
print(f"🔄 Resuming from row: {start_from}\n")


# ------------------------------
# build text with correct fields
# ------------------------------
def build_text(row):
    attack = row.get("label", "Unknown Attack")

    text_list = [
        f"Attack Type: {attack}",
        f"Flow Duration: {row.get('flow_duration', 'N/A')}",
        f"Variance: {row.get('Variance', 'N/A')}",
        f"Weight: {row.get('Weight', 'N/A')}",
        f"Header Length: {row.get('Header_Length', 'N/A')}",
        f"Protocol Type: {row.get('Protocol Type', 'N/A')}",
        f"Duration: {row.get('Duration', 'N/A')}",
        f"Rate: {row.get('Rate', 'N/A')}",
        f"Srate: {row.get('Srate', 'N/A')}",
        f"Drate: {row.get('Drate', 'N/A')}",
    ]

    return "\n".join(text_list)


# ------------------------------
# embed with Gemini
# ------------------------------
def embed_text(text):
    for attempt in range(5):
        try:
            time.sleep(1.6)  # required delay
            result = genai.embed_content(
                model="models/text-embedding-004",
                content=text
            )
            return result["embedding"]
        except Exception as e:
            print(f"Embed failed on attempt {attempt+1}, retrying...")
            time.sleep(3)

    raise Exception("Gemini failed after 5 retries")


# ------------------------------
# upload in batches
# ------------------------------
BATCH = 100
vectors = []

print("🚀 Reindexing started...\n")

# 🔥 Resume loop
for i, row in tqdm(df.iloc[start_from:].iterrows(), total=len(df) - start_from):

    text = build_text(row)
    emb = embed_text(text)

    vectors.append({
        "id": f"row-{i}",
        "values": emb,
        "metadata": {"row": int(i), "text": text, "label": row.get("label")}
    })

    if len(vectors) >= BATCH:
        index.upsert(vectors)
        vectors = []

# upload leftover
if vectors:
    index.upsert(vectors)

print("\n🎉 Reindexing completed successfully!")


📌 Loaded 5001 rows from CSV

🔄 Resuming from row: 1500

🚀 Reindexing started...



100%|██████████| 3501/3501 [1:50:20<00:00,  1.89s/it]  



🎉 Reindexing completed successfully!


In [ ]:
import os
from pinecone import Pinecone
from dotenv import load_dotenv

load_dotenv()

PINECONE_KEY = os.getenv("PINECONE_API_KEY")
INDEX_NAME = os.getenv("INDEX_NAME")

pc = Pinecone(api_key=PINECONE_KEY)
index = pc.Index(INDEX_NAME)

print("⚠️ Deleting all vectors from Pinecone index:", INDEX_NAME)
index.delete(delete_all=True)
print("🧹 All vectors deleted successfully.")